In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# 🎙️ Llama 3 Professional Analyst Station — Kaggle T4
# Исправленная и оптимизированная версия
# ============================================================

# --- УСТАНОВКА ЗАВИСИМОСТЕЙ ---
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6
!pip install -q deep-translator gradio
!pip install -q git+https://github.com/openai/whisper.git


In [ ]:
# !pip install -q "pyannote.audio==2.1.1"
# !pip install -q "torchaudio==2.0.2" --force-reinstall

In [ ]:
# !pip install -q "numpy==2.0.2" --force-reinstall
# !pip install -q "numba==0.60.0" --force-reinstall

In [ ]:
# --- ИМПОРТЫ ---
import os
import time
import torch
import whisper
import gradio as gr

from threading import Thread
from concurrent.futures import ThreadPoolExecutor

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TextIteratorStreamer,
)
from deep_translator import GoogleTranslator
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
# from pyannote.audio import Pipeline
# print("✅ pyannote загружен успешно")

In [ ]:
# ============================================================
# --- КОНСТАНТЫ ---
# ============================================================
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
WHISPER_MODEL_SIZE = "small"   # base / small / medium — на T4 "small" оптимален
MAX_NEW_TOKENS_GRAMMAR = 2500
MAX_NEW_TOKENS_ANALYSIS = 1200
TRANSLATION_INTERVAL_SEC = 1.5


In [ ]:
ROLES = {
    "📊 Аналитик":    "You are a professional analyst. Extract core essence and key points. Use Markdown.",
    "⚔️ Критик":      "You are a critical opponent. Find weak arguments, logical flaws and inconsistencies.",
    "📱 SMM-менеджер":"You are an SMM manager. Create an engaging Telegram post based on this content.",
    "📋 Секретарь":   "You are a secretary. Create a meeting protocol: who said what, what tasks were assigned.",
    "🧠 Психолог":    "You are a psychologist. Analyze the emotional tone, stress points and speaker's mental state.",
    "🎯 Коуч":        "You are a life coach. Extract actionable insights and next steps from this content.",
}

In [ ]:
# ============================================================
# --- АВТОРИЗАЦИЯ HuggingFace ---
# ============================================================
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("✅ HuggingFace авторизация успешна")
else:
    raise RuntimeError("❌ Секрет HF_TOKEN не найден. Добавьте его в Add-ons -> Secrets")

In [ ]:
# ============================================================
# --- ПРОВЕРКА АУДИОФАЙЛА (опционально) ---
# ============================================================
audio_filename = "/kaggle/input/audiotest/denver_extract.mp3"

if os.path.exists(audio_filename):
    print(f"✅ Файл {audio_filename} найден")
else:
    print(f"⚠️  Файл не найден: {audio_filename}")
    available = os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else []
    print("Доступные датасеты:", available)


In [ ]:

# ============================================================
# --- ЗАГРУЗКА МОДЕЛЕЙ ---
# ============================================================
print(f"🔄 Загружаем Whisper '{WHISPER_MODEL_SIZE}'...")
stt_model = whisper.load_model(WHISPER_MODEL_SIZE)
print("✅ Whisper загружен")


# print("🔄 Загружаем Speaker Diarization...")
# diarization_pipeline = Pipeline.from_pretrained(
#     "pyannote/speaker-diarization-3.1",
#     use_auth_token=hf_token
# )
# diarization_pipeline.to(torch.device("cuda"))
# print("✅ Speaker Diarization загружен")

In [ ]:
# ============================================================
# --- ЗАГРУЗКА МОДЕЛЕЙ ---
# ============================================================
print(f"🔄 Загружаем Whisper '{WHISPER_MODEL_SIZE}'...")
stt_model = whisper.load_model(WHISPER_MODEL_SIZE)
print("✅ Whisper загружен")

In [ ]:
# 4-bit квантование для экономии VRAM на T4 (16 GB)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)


In [ ]:
print(f"🔄 Загружаем {LLAMA}...")
tokenizer = AutoTokenizer.from_pretrained(LLAMA, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.eval()  # отключаем dropout — ускоряет инференс
print("✅ LLaMA загружена")

In [ ]:
# Executor для асинхронного перевода (не блокирует стриминг)
_translator_executor = ThreadPoolExecutor(max_workers=1)

# ============================================================
# --- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---
# ============================================================

def _translate_async(text: str) -> str:
    try:
        return GoogleTranslator(source="en", target="ru").translate(text)
    except Exception:
        return text

def _clear_cuda_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# --- ЛОГИКА ОБРАБОТКИ ---
# ============================================================

def transcribe_raw(audio_path: str) -> str:
    if audio_path is None:
        return ""
    result = stt_model.transcribe(audio_path, fp16=True)
    return result["text"]


# def transcribe_with_speakers(audio_path: str) -> str:
#     """Расшифровка с определением спикеров через pyannote."""
#     if audio_path is None:
#         return ""

# def transcribe_auto(audio_path: str, mode: str) -> str:
#     """Выбирает режим расшифровки по переключателю."""
#     if mode == "👥 Определить спикеров":
#         return transcribe_with_speakers(audio_path)
#     return transcribe_raw(audio_path)


    
#     # Шаг 1 — определяем кто говорит и когда
#     diarization = diarization_pipeline(audio_path)

#     # Шаг 2 — полная расшифровка через Whisper с таймкодами
#     whisper_result = stt_model.transcribe(
#         audio_path,
#         fp16=True,
#         word_timestamps=True,
#     )

#     # Шаг 3 — совмещаем сегменты Whisper с метками спикеров
#     output_lines = []

#     for segment in whisper_result["segments"]:
#         seg_start = segment["start"]
#         seg_end   = segment["end"]
#         seg_text  = segment["text"].strip()

#         # Находим спикера который говорил большую часть этого сегмента
#         speaker = "Спикер ?"
#         max_overlap = 0.0

#         for turn, _, spk in diarization.itertracks(yield_label=True):
#             overlap = min(turn.end, seg_end) - max(turn.start, seg_start)
#             if overlap > max_overlap:
#                 max_overlap = overlap
#                 # Форматируем: SPEAKER_00 → Спикер 1
#                 num = int(spk.split("_")[-1]) + 1
#                 speaker = f"Спикер {num}"

#         # Форматируем таймкод [MM:SS]
#         mins = int(seg_start) // 60
#         secs = int(seg_start) % 60
#         timestamp = f"[{mins:02d}:{secs:02d}]"

#         output_lines.append(f"**{speaker}** {timestamp}: {seg_text}")

#     return "\n\n".join(output_lines)


def fix_grammar_llama(raw_text: str) -> str:
    if not raw_text.strip():
        return "### ⚠️ Нет текста для обработки."

    prompt = (
        "Fix grammar and punctuation only. "
        "Keep all details and original meaning exactly. "
        "Return only the corrected text without any commentary:\n\n"
        f"{raw_text}"
    )
    messages = [{"role": "user", "content": prompt}]

    encoded = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    )
    input_ids = encoded.to("cuda")
    attention_mask = torch.ones_like(input_ids).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS_GRAMMAR,
            use_cache=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    result = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True,
    )
    _clear_cuda_cache()
    return result


# ← ИСПРАВЛЕНО: добавлен параметр role
def generate_analysis_stream(user_text: str, custom_query: str, target_lang: str, role: str):
    if not user_text.strip():
        yield "### ⚠️ Текст отсутствует. Сначала расшифруйте аудио."
        return

    # Берём system prompt из словаря по выбранной роли
    system_instr = ROLES.get(role, ROLES["📊 Аналитик"])

    if custom_query.strip():
        # Если есть вопрос — роль игнорируется, отвечаем на вопрос
        system_instr = (
            "You are a precise assistant. "
            "Answer the user's SPECIFIC QUESTION based on the provided transcript. "
            "Give a direct, concise answer only."
        )
        user_task = f"Answer this question: '{custom_query}'\n\nTRANSCRIPT:\n{user_text}"
    else:
        user_task = f"Provide your analysis.\n\nTRANSCRIPT:\n{user_text}"

    messages = [
        {"role": "system", "content": system_instr},
        {"role": "user",   "content": user_task},
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    )
    input_ids = encoded.to("cuda")
    attention_mask = torch.ones_like(input_ids).to("cuda")

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    gen_kwargs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "streamer": streamer,
        "max_new_tokens": MAX_NEW_TOKENS_ANALYSIS,
        "temperature": 0.4,
        "do_sample": True,
        "use_cache": True,
        "repetition_penalty": 1.1,
        "pad_token_id": tokenizer.eos_token_id,
    }

    thread = Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()

    full_en = ""
    tr_cache = ""
    last_tr_time = time.time()
    tr_future = None

    for new_chunk in streamer:
        full_en += new_chunk

        if target_lang == "en":
            yield full_en + " ▌"
            continue

        now = time.time()
        sentence_end = any(c in new_chunk for c in ".!?\n")
        should_translate = (now - last_tr_time > TRANSLATION_INTERVAL_SEC) or sentence_end

        if should_translate and (tr_future is None or tr_future.done()):
            tr_future = _translator_executor.submit(_translate_async, full_en)
            last_tr_time = now

        if tr_future is not None and tr_future.done():
            try:
                tr_cache = tr_future.result()
            except Exception:
                pass

        yield (tr_cache if tr_cache else full_en) + " ▌"

    if target_lang != "en" and full_en:
        try:
            final_translation = GoogleTranslator(source="en", target="ru").translate(full_en)
            yield final_translation
        except Exception:
            yield full_en
    else:
        yield full_en

    thread.join()
    _clear_cuda_cache()


def save_result(text: str, mode: str):
    if not text or not text.strip() or text.startswith("### 🤖"):
        return None
    ext = "txt" if mode == "full" else "md"
    fname = f"/kaggle/working/result_{mode}_{int(time.time())}.{ext}"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(text)
    return fname


# ============================================================
# --- GRADIO UI ---
# ============================================================

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&family=Syne:wght@400;700;800&display=swap');

* { font-family: 'Syne', sans-serif !important; }
code, .mono { font-family: 'JetBrains Mono', monospace !important; }

body, .gradio-container {
    background: #0d0f14 !important;
}

.output-box {
    background: #12151d !important;
    color: #e8eaf0 !important;
    border: 1px solid #ff6600 !important;
    border-radius: 8px;
    padding: 20px;
    min-height: 420px;
    font-size: 0.95rem;
    line-height: 1.7;
}
.output-box * { color: #e8eaf0 !important; }
.output-box h1, .output-box h2, .output-box h3 { color: #ff6600 !important; }

#query-input label span { color: #ff9944 !important; font-weight: 700; }

.gr-button-primary {
    background: linear-gradient(135deg, #ff6600, #ff9900) !important;
    border: none !important;
    font-weight: 700 !important;
}
.gr-button-secondary {
    border: 1px solid #ff6600 !important;
    color: #ff6600 !important;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="orange")) as demo:

    gr.Markdown(
        "# 🎙️ Llama 3 · Professional Analyst Station\n"
        "*Kaggle T4 Edition — Whisper + LLaMA 3.2 3B Instruct*"
    )

    with gr.Row():

        # ---- ЛЕВАЯ КОЛОНКА: ввод и расшифровка ----
        with gr.Column(scale=1):
            with gr.Tabs():
                with gr.TabItem("🎤 Микрофон"):
                    mic_in = gr.Audio(
                        sources="microphone",
                        type="filepath",
                        label=None,
                    )
                with gr.TabItem("📁 Загрузить файл"):
                    file_in = gr.Audio(
                        sources="upload",
                        type="filepath",
                        label=None,
                    )

            mode_sel = gr.Radio(
                    choices=["🎙️ Обычная расшифровка", "👥 Определить спикеров"],
                    value="🎙️ Обычная расшифровка",
                    label="Режим расшифровки",
                    )

            
            txt_raw = gr.Textbox(
                label="📝 Расшифровка (Whisper)",
                lines=12,
                show_copy_button=True,
                placeholder="Здесь появится текст после расшифровки...",
            )

            with gr.Row():
                btn_fix = gr.Button("✏️ Исправить грамматику (LLaMA)", variant="secondary")
                btn_dl_full = gr.Button("💾 Скачать .txt", variant="primary")

            out_full = gr.File(label="Файл расшифровки (.txt)")

        # ---- ПРАВАЯ КОЛОНКА: анализ ----
        with gr.Column(scale=1):

            query_in = gr.Textbox(
                label="❓ Ваш вопрос к тексту",
                placeholder="Введите вопрос или оставьте пустым для авто-суммаризации...",
                lines=2,
                elem_id="query-input",
            )

            # ← ДОБАВЛЕН role_sel
            role_sel = gr.Dropdown(
                choices=list(ROLES.keys()),
                value="📊 Аналитик",
                label="🎭 Роль аналитика",
            )

            with gr.Row():
                lang_sel = gr.Radio(
                    choices=["ru", "en"],
                    value="ru",
                    label="🌍 Язык ответа",
                    scale=1,
                )
                btn_run = gr.Button("🚀 Запустить анализ", variant="primary", scale=2)

            out_md = gr.Markdown(
                value="### 🤖 Ожидание запроса...",
                elem_classes="output-box",
            )

            with gr.Row():
                btn_dl_sum = gr.Button("💾 Скачать анализ .md", variant="primary")
                btn_clr = gr.Button("🧹 Сбросить всё", variant="stop")

            out_sum = gr.File(label="Анализ (.md)")

    # ---- СОБЫТИЯ ----
    mic_in.change(fn=transcribe_raw, inputs=mic_in, outputs=txt_raw)
    file_in.change(fn=transcribe_raw, inputs=file_in, outputs=txt_raw)
    btn_fix.click(fn=fix_grammar_llama, inputs=txt_raw, outputs=txt_raw)
    btn_dl_full.click(fn=lambda x: save_result(x, "full"), inputs=txt_raw, outputs=out_full)
    btn_run.click(
        fn=generate_analysis_stream,
        inputs=[txt_raw, query_in, lang_sel, role_sel],  # ← role_sel добавлен
        outputs=out_md,
    )
    btn_dl_sum.click(fn=lambda x: save_result(x, "summary"), inputs=out_md, outputs=out_sum)
    btn_clr.click(
        fn=lambda: (None, None, "", "", "### 🤖 Ожидание...", None, None),
        outputs=[mic_in, file_in, txt_raw, query_in, out_md, out_full, out_sum],
    )

# ============================================================
# --- ЗАПУСК ---
# ============================================================
demo.queue().launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
    inline=True,
    debug=True,
)